# ch06 Bonus 03：Gradio 情感分类交互界面

> 对照官方 `ch06/04_user_interface`

## 一句话

给训练好的情感分类器套一个 **Gradio 网页 UI**：用户在浏览器输入一句话，实时显示「正面/负面」判断和置信度。

## 原理

把 `classify(text) → (label, confidence)` 函数用 `gr.Interface` 包装成 web 服务。LLM 分类场景下，输入是文本框，输出是标签 + 概率条。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 1. 先训练一个 demo 分类器（复用主线流程）
tok = tiktoken.get_encoding("gpt2")
POS = ["这部电影非常精彩 我很喜欢", "太好看了 剧情感人至深", "画面优美 值得推荐",
       "演技出色 故事动人", "完美之作 强烈推荐", "音乐动听 视觉震撼",
       "节奏紧凑 引人入胜", "结局温暖 回味无穷"]
NEG = ["太糟糕了 浪费时间", "剧情无聊 让人失望", "画面粗糙 毫无诚意",
       "演技尴尬 故事混乱", "简直烂片 不忍直视", "噪音刺耳 看不下去",
       "节奏拖沓 昏昏欲睡", "结局糟糕 一无是处"]
texts = POS + NEG
labels = [1]*len(POS) + [0]*len(NEG)

def make_dataset(texts, labels, max_len=32):
    data = [(tok.encode(t)[:max_len] + [50256]*(max_len-len(tok.encode(t)[:max_len])), l)
            for t, l in zip(texts, labels)]
    return data

data = make_dataset(texts, labels)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim":128, "n_layers":2, "n_heads":4, "context_length":32})
torch.manual_seed(123)
model = GPTModel(cfg)
model.out_head = nn.Linear(cfg["emb_dim"], 2)
for p in model.parameters(): p.requires_grad = False
for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
for p in model.final_norm.parameters(): p.requires_grad = True
for p in model.out_head.parameters(): p.requires_grad = True
model.to(device)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)
model.train()
for _ in range(15):
    for ids, l in data:
        x = torch.tensor([ids]).to(device); y = torch.tensor([l]).to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(x)[:, -1, :], y)
        loss.backward(); opt.step()
model.eval()
print("✓ demo 分类器已训练")

In [ ]:
def classify_sentiment(text):
    """Gradio 包装的函数：输入文本，返回标签+置信度。"""
    max_len = 32
    ids = tok.encode(text)[:max_len]
    ids = ids + [50256]*(max_len-len(ids))
    with torch.no_grad():
        logits = model(torch.tensor([ids]).to(device))[:, -1, :]
        probs = F.softmax(logits, dim=-1)[0]
    neg, pos = probs[0].item(), probs[1].item()
    label = "正面 😊" if pos > neg else "负面 😞"
    return {label: max(pos, neg)}, {"正面": pos, "负面": neg}

# 测试函数
for t in ["这部电影太精彩了", "简直浪费时间烂片"]:
    result, probs = classify_sentiment(t)
    print(f"{t!r:25} → {result} 概率 {probs}")

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=classify_sentiment,
    inputs=gr.Textbox(label="输入影评", placeholder="输入一句影评...",
                      value="这部电影非常精彩"),
    outputs=[
        gr.Label(label="预测结果"),
        gr.Label(label="置信度分布"),
    ],
    title="影评情感分类",
    description="输入一句话，判断正面/负面。（demo 用未训练小模型，加载预训练权重后更准）",
)

print("Gradio 界面已定义。")
print("启动方式（终端）: demo.launch() → http://127.0.0.1:7860")
print("\n这里不实际 launch（会阻塞 notebook）；取消注释下行启动：")
# demo.launch()  # 取消注释以启动服务